# FileFormats

The `fileformats` library provides a typed hierarchy of file-format classes. In the
AIS Pipelines framework, format types serve two purposes:

1. **Type-safe data handoffs** — the output type of one pipeline step must be compatible
   with the input type of the next, checked before the pipeline runs.
2. **Shell task templates** — MIME-like strings in task definitions (e.g.
   `<in_file:medimage/nifti1>`) resolve to specific format classes at runtime.

This workbook covers:
- Using existing format types
- Defining a simple custom binary format
- Validated formats with magic numbers
- Multi-file formats (binary data + separate header)
- A real-world example: Bruker ParaVision

In [ ]:
%pip install -qr ../requirements.txt

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

test_dir = Path(mkdtemp())

## 1. Using existing format types

Format classes are organised into sub-packages by domain — `fileformats.generic` for
unconstrained types, and domain-specific packages such as `fileformats.medimage`,
`fileformats.image`, and `fileformats.application`. Instantiating a format object
validates the path immediately.

In [ ]:
from fileformats.medimage import Nifti1
from fileformats.application import Zip
from fileformats.generic import BinaryFile, File

nifti = Nifti1.sample()  # generates a temporary test file

print(type(nifti))
print(nifti.fspath)

### MIME-like strings

Every format class has a unique MIME or MIME-like identifier. Standard formats use
official IANA MIME types (e.g. `application/zip`); domain-specific formats use a
MIME-like namespace derived from the Python sub-package underneath `fileformats`
(e.g. `fileformats.medimage.Nifti1` maps to `medimage/nifti1`).

These strings appear in pipelines framework YAML specifications —
so knowing a format's MIME-like string is how you reference it from outside Python.

In [ ]:
print(Nifti1.mime_like)     # medimage/nifti1
print(Zip.mime_like)        # application/zip
print(BinaryFile.mime_like) # generic/binary-file

In [ ]:
from fileformats.core.identification import from_mime

# Resolve a string back to its class — this is what pydra does internally
FormatClass = from_mime("medimage/nifti1")
print(FormatClass)

### Type hierarchy

Format classes form an inheritance hierarchy. `Nifti1` is a `BinaryFile` is a `File`
is a `FileSet`. This matters in pydra workflows: if a task output is typed `Nifti1`
and the next input expects `BinaryFile`, the assignment passes — but passing a
`Jpeg` where a `Nifti1` is expected raises a `TypeError` before any data moves.

In [ ]:
from fileformats.core import FileSet

print(issubclass(Nifti1, BinaryFile))  # True
print(issubclass(Nifti1, File))        # True
print(issubclass(Nifti1, FileSet))     # True
print(issubclass(BinaryFile, Nifti1))  # False

### `mock()` and `sample()`

`mock()` returns a typed placeholder that passes type-checking but points to no real
file — useful for inspecting a shell task's command line without running anything.

`sample()` generates a real temporary file of the correct format — useful for testing
a task end-to-end.

In [ ]:
# mock — typed placeholder, no file on disk
mock_nifti = Nifti1.mock()
print(f"mock path:   {mock_nifti.fspath}")
print(f"file exists: {mock_nifti.fspath.exists()}")

# sample — real file on disk
real_nifti = Nifti1.sample(test_dir, seed=0)
print(f"\nsample path: {real_nifti.fspath}")
print(f"file exists: {real_nifti.fspath.exists()}")

## 2. Defining a simple binary format

The minimal custom format is a `BinaryFile` subclass with an `ext` attribute. This
is enough for fileformats to validate that a path has the correct suffix.

> **Namespace constraint:** Format classes must be defined within the `fileformats.*`
> namespace. In a production package this happens automatically. In this workbook we
> set `__module__` manually after each class definition.

In [ ]:
from fileformats.generic import BinaryFile


class MyBinaryFormat(BinaryFile):
    """A simple custom binary format identified only by its extension."""

    ext = ".mbf"


MyBinaryFormat.__module__ = "fileformats.workshop"

mbf_path = test_dir / "data.mbf"
mbf_path.write_bytes(b"\x00" * 256)

f = MyBinaryFormat(mbf_path)
print(f"Loaded: {f.fspath}")
print(f"MIME-like: {MyBinaryFormat.mime_like}")

In [ ]:
from fileformats.core.exceptions import FormatMismatchError

wrong_path = test_dir / "data.txt"
wrong_path.write_text("not a .mbf file")

try:
    MyBinaryFormat(wrong_path)
except FormatMismatchError as e:
    print(f"Rejected: {e}")

## 3. Validated formats with magic numbers

Many binary formats begin with a fixed byte sequence — a *magic number* — that
identifies the format independently of the file extension. The `WithMagicNumber`
mixin adds this check automatically at load time. The magic number is given as a
hex string.

Here we define a toy image format with a 4-byte header:

In [ ]:
import struct
import numpy as np
from fileformats.core.mixin import WithMagicNumber
from fileformats.generic import BinaryFile


class MyImageFormat(WithMagicNumber, BinaryFile):
    """A toy single-channel float32 image format.

    File layout:
        [0:4]   magic number  b'MIF\x00'
        [4:8]   width         uint32 little-endian
        [8:12]  height        uint32 little-endian
        [12:]   pixel data    float32 little-endian, row-major
    """

    ext = ".mif"
    magic_number = "4d494600"  # b'MIF\x00' as hex

    @property
    def shape(self):
        w, h = struct.unpack_from("<II", self.fspath.read_bytes(), offset=4)
        return (h, w)

    def read_pixels(self) -> np.ndarray:
        h, w = self.shape
        return np.frombuffer(self.fspath.read_bytes()[12:], dtype="<f4").reshape(h, w)


MyImageFormat.__module__ = "fileformats.workshop"

In [ ]:
def write_mif(path: Path, pixels: np.ndarray) -> None:
    h, w = pixels.shape
    with open(path, "wb") as f:
        f.write(b"MIF\x00")
        f.write(struct.pack("<II", w, h))
        f.write(pixels.astype("<f4").tobytes())


rng = np.random.default_rng(seed=0)
pixels = rng.random((64, 64), dtype=np.float32)

mif_path = test_dir / "image.mif"
write_mif(mif_path, pixels)

img = MyImageFormat(mif_path)
print(f"Shape: {img.shape}")
print(f"Values match: {np.allclose(img.read_pixels(), pixels)}")

In [ ]:
# A file with the right extension but wrong magic number is rejected
bad_path = test_dir / "bad.mif"
bad_path.write_bytes(b"XXXX" + b"\x00" * 100)

try:
    MyImageFormat(bad_path)
except FormatMismatchError as e:
    print(f"Rejected: {e}")

## 4. Multi-file formats (binary data + separate header)

Many scientific formats split metadata and data across two files with the same stem
but different extensions — for example `.hdr`/`.img` (Analyze) or `.mih`/`.dat`
(MRTrix3 image header format). The `WithSeparateHeader` mixin handles this: providing
just the primary file path is enough to locate both files automatically.

Here we implement a toy paired format: a plain-text key-value header (`.khdr`) and
a raw float32 data file (`.kdat`).

In [ ]:
from fileformats.generic import BinaryFile, UnicodeFile
from fileformats.core.mixin import WithSeparateHeader


class KeyValueHeader(UnicodeFile):
    """Plain-text header: one 'key = value' pair per line."""

    ext = ".khdr"


class KeyValueData(WithSeparateHeader, BinaryFile):
    """Raw float32 data array with an adjacent text header."""

    ext = ".kdat"
    header_type = KeyValueHeader

    def read_metadata(self, **kwargs):
        meta = {}
        for line in self.header.fspath.read_text().splitlines():
            if "=" in line:
                key, _, val = line.partition("=")
                meta[key.strip()] = val.strip()
        return meta

    @property
    def shape(self):
        return tuple(int(d) for d in self.metadata["shape"].split(","))

    def read_array(self) -> np.ndarray:
        return np.frombuffer(self.fspath.read_bytes(), dtype="<f4").reshape(self.shape)


KeyValueHeader.__module__ = "fileformats.workshop"
KeyValueData.__module__ = "fileformats.workshop"

In [ ]:
array = rng.random((32, 32, 16)).astype(np.float32)

(test_dir / "volume.khdr").write_text(
    f"shape = {','.join(str(d) for d in array.shape)}\n"
    "dtype = float32\n"
    "endian = little\n"
)
(test_dir / "volume.kdat").write_bytes(array.astype("<f4").tobytes())

# Providing just the .kdat path — the .khdr is found automatically from the same stem
volume = KeyValueData(test_dir / "volume.kdat")

print(f"Files: {[p.name for p in volume.fspaths]}")
print(f"Shape: {volume.shape}")
print(f"Values match: {np.allclose(volume.read_array(), array)}")

### Implementing `generate_sample_data`

Registering a `generate_sample_data` implementation with `@extra_implementation`
enables `MyFormat.sample()` to work, which in turn allows pydra tasks to generate
test instances of the format in automated tests.

The function receives a `SampleFileGenerator` with `dest_dir`, `seed`, and
`fname_stem` attributes, and must return a list of the paths it creates.

In [ ]:
import typing as ty
from fileformats.core import FileSet, SampleFileGenerator, extra_implementation


@extra_implementation(FileSet.generate_sample_data)
def generate_key_value_data(
    self: KeyValueData, generator: SampleFileGenerator
) -> ty.List[Path]:
    rng = np.random.default_rng(generator.seed)
    array = rng.random((16, 16, 8)).astype(np.float32)

    hdr = generator.dest_dir / (generator.fname_stem + ".khdr")
    dat = generator.dest_dir / (generator.fname_stem + ".kdat")

    hdr.write_text(
        f"shape = {','.join(str(d) for d in array.shape)}\n"
        "dtype = float32\n"
    )
    dat.write_bytes(array.astype("<f4").tobytes())
    return [hdr, dat]


sample = KeyValueData.sample(seed=42)
print(f"Shape: {sample.shape}")

## 5. Real-world example: Bruker ParaVision

Bruker's ParaVision MRI software stores reconstructed images as a pair of extensionless
files that always have fixed names:

- `2dseq` — raw binary voxel data (signed 16-bit integers)
- `visu_pars` — image parameters in JCAMP-DX text format

Both live in the same scan reconstruction directory. Because neither file has an
extension, `WithSeparateHeader` cannot distinguish them — it relies on extensions to
locate the header file. Instead we use the base `FileSet` class and use
`validated_property` to locate each file by name and assert both are present.

In [ ]:
import re


def parse_jcamp_dx(text: str) -> dict:
    """Parse a Bruker JCAMP-DX parameter file into a plain dictionary.

    Handles scalar integers/floats and simple numeric arrays.
    Array-size prefixes like '( 1, 3 )' are stripped automatically.
    """
    params: dict = {}
    current_key: ty.Optional[str] = None
    value_lines: ty.List[str] = []

    def flush() -> None:
        if current_key is None:
            return
        raw = re.sub(r"^\(.*?\)\s*", "", " ".join(value_lines).strip())
        parts = raw.split()
        if not parts:
            return
        if len(parts) == 1:
            try:
                params[current_key] = int(parts[0])
            except ValueError:
                try:
                    params[current_key] = float(parts[0])
                except ValueError:
                    params[current_key] = parts[0].strip("<>")
        else:
            try:
                params[current_key] = [int(p) for p in parts]
            except ValueError:
                try:
                    params[current_key] = [float(p) for p in parts]
                except ValueError:
                    params[current_key] = parts

    for line in text.splitlines():
        if line.startswith("##$"):
            flush()
            current_key, _, rest = line[3:].partition("=")
            current_key = current_key.strip()
            value_lines = [rest.strip()]
        elif line.startswith("##"):
            flush()
            current_key = None
            value_lines = []
        elif current_key:
            value_lines.append(line.strip())

    flush()
    return params

In [ ]:
from fileformats.core import FileSet
from fileformats.core.decorators import validated_property
from fileformats.core.exceptions import FormatMismatchError


class TwoDSeq(FileSet):
    """Bruker ParaVision reconstructed image: '2dseq' data + 'visu_pars' parameters."""

    @validated_property
    def data_fspath(self) -> Path:
        """Path to the 2dseq binary data file."""
        matches = [p for p in self.fspaths if p.name == "2dseq"]
        if not matches:
            raise FormatMismatchError(
                f"No '2dseq' file found in {sorted(self.fspaths)}"
            )
        return matches[0]

    @validated_property
    def params_fspath(self) -> Path:
        """Path to the visu_pars JCAMP-DX parameter file."""
        matches = [p for p in self.fspaths if p.name == "visu_pars"]
        if not matches:
            raise FormatMismatchError(
                f"No 'visu_pars' file found in {sorted(self.fspaths)}"
            )
        return matches[0]

    def read_metadata(self, **kwargs) -> dict:
        return parse_jcamp_dx(self.params_fspath.read_text())

    @property
    def shape(self) -> tuple:
        """Image matrix dimensions (Nx, Ny, Nz) from visu_pars."""
        return tuple(self.metadata["VisuCoreSize"])

    @property
    def voxel_size_mm(self) -> tuple:
        """Voxel dimensions in mm (dx, dy, dz) from visu_pars."""
        return tuple(self.metadata["VisuCoreVoxelDim"])

    def read_array(self) -> np.ndarray:
        """Read voxel intensities as a float32 array, applying the slope from visu_pars."""
        slope = self.metadata.get("VisuCoreDataSlope", 1.0)
        if isinstance(slope, list):
            slope = slope[0]
        raw = np.frombuffer(self.data_fspath.read_bytes(), dtype="<i2")
        return (raw.astype(np.float32) * slope).reshape(self.shape)


TwoDSeq.__module__ = "fileformats.vendor.bruker"

In [ ]:
def write_paravision_scan(
    dest_dir: Path,
    shape: tuple = (128, 128, 32),
    voxel_mm: tuple = (0.156, 0.156, 0.5),
    seed: int = 0,
) -> Path:
    """Write a minimal synthetic Bruker ParaVision reconstruction directory."""
    dest_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)

    data = (rng.random(shape) * 4095).astype(np.int16)
    (dest_dir / "2dseq").write_bytes(data.astype("<i2").tobytes())

    ndim = len(shape)
    visu_pars = (
        "##TITLE= VisuPars\n"
        "##JCAMPDX= 4.24\n"
        f"##$VisuCoreSize=( 1, {ndim} )\n"
        + " ".join(str(d) for d in shape) + "\n"
        + f"##$VisuCoreVoxelDim=( 1, {ndim} )\n"
        + " ".join(str(v) for v in voxel_mm) + "\n"
        + "##$VisuCoreDataSlope=( 1, 1 )\n1.0\n"
        + "##$VisuCoreDataOffs=( 1, 1 )\n0.0\n"
        + "##END=\n"
    )
    (dest_dir / "visu_pars").write_text(visu_pars)
    return dest_dir


scan_dir = write_paravision_scan(test_dir / "pv_scan")
print(f"Files: {[f.name for f in scan_dir.iterdir()]}")

In [ ]:
scan = TwoDSeq([scan_dir / "2dseq", scan_dir / "visu_pars"])

print(f"Shape:      {scan.shape}")
print(f"Voxel (mm): {scan.voxel_size_mm}")

array = scan.read_array()
print(f"Array shape: {array.shape}")
print(f"Value range: [{array.min():.1f}, {array.max():.1f}]")

In [ ]:
# A file set missing visu_pars is rejected at instantiation
try:
    TwoDSeq([scan_dir / "2dseq"])
except FormatMismatchError as e:
    print(f"Rejected: {e}")

### Publishing a format as a package

The `fileformats-vendor-bruker` package is the intended home for `TwoDSeq`. Format
packages follow the naming convention `fileformats-<domain>` (for domain-specific
formats like `fileformats-medimage`) or `fileformats-vendor-<name>` (for vendor formats
like `fileformats-vendor-bruker`). Classes install into `fileformats.<domain>` or
`fileformats.vendor.<name>` respectively — which is also where their MIME-like string
comes from.

Once published, the format can be referenced by its MIME-like string anywhere in the
framework:

```python
from pydra.compose import shell

PvReco = shell.define(
    "pv-reco <scan:medimage/vnd.bruker.two-d-seq> <out|nifti:medimage/nifti1>"
)
```

## 6. Extra hooks and converters

FileFormats has two complementary extension mechanisms that let behaviour live outside
the core format class — useful when an implementation depends on a heavy optional
dependency, or when the format is defined in one package and the I/O logic in another.

### Extra hooks

Methods marked with `@extra` on `FileSet` are *dispatch points*: they raise
`NotImplementedError` until a concrete implementation is registered elsewhere with
`@extra_implementation`. The three most important hooks are:

| Hook | Purpose |
|------|---------|
| `FileSet.load` | Load file contents into a Python object (numpy array, dict, …) |
| `FileSet.save` | Write a Python object back to the file |
| `FileSet.read_metadata` | Return a `{key: value}` metadata mapping |
| `FileSet.generate_sample_data` | Generate a synthetic test file (enables `.sample()`) |

You can override `read_metadata` and `generate_sample_data` *directly* in your class
when you know the implementation upfront (as in sections 3–5). Use
`@extra_implementation` when the implementation belongs in a separate package — for
example a `fileformats-extras` add-on that pulls in `nibabel` or `Pillow`.

### Converters

A pydra task decorated with `@converter` is registered as a conversion route between
two format types. Once registered, `TargetFormat.convert(source_instance)` anywhere in
the framework automatically runs the task and wraps its output in the target type.

### Implementing `load` and `save`

The `load` and `save` hooks for `MyImageFormat` are natural: `load` returns a numpy
array and `save` writes one back. Because these hooks live on `FileSet` as `@extra`
dispatch points, we register our implementations with `@extra_implementation` rather
than overriding inside the class.

In [ ]:
import typing as ty
from fileformats.core import FileSet, extra_implementation


@extra_implementation(FileSet.load)
def load_my_image(self: MyImageFormat, **kwargs: ty.Any) -> ty.Any:
    return self.read_pixels()


@extra_implementation(FileSet.save)
def save_my_image(self: MyImageFormat, data: ty.Any, **kwargs: ty.Any) -> None:
    write_mif(self.fspath, data)


# Round-trip test: load, modify, save, reload
img = MyImageFormat(mif_path)
loaded = img.load()
print(f"Loaded shape: {loaded.shape}, dtype: {loaded.dtype}")

brightened = loaded * 2.0
img.save(brightened)

reloaded = MyImageFormat(mif_path).load()
print(f"Values doubled: {np.allclose(reloaded, brightened)}")

### Registering a converter

A converter is a pydra task that transforms one format into another. Decorating it
with `@converter` registers the (source → target) route globally.

`@converter` infers the source format from the type annotation of the `in_file`
parameter and the target format from the `outputs` dict passed to `@python.define`.
You can also pass `source_format=` / `target_format=` explicitly when the types cannot
be inferred, or when the same task handles multiple routes (see the image converter in
`fileformats-extras` for an example of stacking multiple `@converter` decorators).

In [ ]:
from pydra.compose import python
from fileformats.core import converter


# Convert a 2-D MyImageFormat into a 1-slice KeyValueData volume.
@converter
@python.define(outputs={"out_file": KeyValueData})  # type: ignore[untyped-decorator]
def mif_to_kv(in_file: MyImageFormat) -> KeyValueData:
    pixels = in_file.read_pixels()          # (H, W) float32
    shape = (*pixels.shape, 1)              # treat the 2-D slice as a 1-slice volume
    out_dir = Path(mkdtemp())
    hdr = out_dir / "slice.khdr"
    dat = out_dir / "slice.kdat"
    hdr.write_text(f"shape = {','.join(str(d) for d in shape)}\ndtype = float32\n")
    dat.write_bytes(pixels.reshape(shape).astype("<f4").tobytes())
    return KeyValueData(dat)


print("Converter registered:", KeyValueData.get_converter(MyImageFormat))

In [ ]:
img = MyImageFormat(mif_path)

# TargetFormat.convert(source) runs the registered task and returns a typed result
kv = KeyValueData.convert(img)

print(f"Source format : {type(img).__name__}  shape={img.shape}")
print(f"Target format : {type(kv).__name__}   shape={kv.shape}")
print(f"Pixel values  : {np.allclose(kv.read_array()[..., 0], img.read_pixels())}")